## Auditoria final

Esse notebook novo (`02_silver_quality_check.py`) serve como uma **auditoria** final da camada Silver inteira: pra cada uma das 6 tabelas que processamos, comparamos:

- Quantas linhas tinha na Bronze vs. quantas sobraram na Silver (quanto foi descartado, e se essa quantidade faz sentido)
- Quantos nulos existiam nas colunas-chave antes vs. depois (deve ir a zero depois)
- Quantas duplicatas existiam antes vs. depois (idem)

**Por que isso é importante**

1. Detecção de regressão - se algum dia a gente alterar a lógica de limpeza (numa Sprint futura, ou revisando o código), esse notebook funciona como um "teste automatizado informal": rodar ele de novo revela na hora se alguma tabela quebrou o padrão esperado.

2. Evidência auditável, num lugar só - em vez de vasculhar 4 PRs diferentes procurando "cadê a prova de que orders ficou limpo", você tem um relatório único, consolidado, que qualquer pessoa (ou você mesma, meses depois) consulta pra confirmar a qualidade de toda a camada de uma vez.

3. Antes de avançar pra Gold, você quer ter certeza - a camada Gold vai fazer JOINs entre essas tabelas. Se alguma tiver um problema de qualidade não detectado, ele se propaga e vira um bug muito mais difícil de rastrear lá na frente. Esse notebook é o "portão de qualidade" antes de seguir adiante.

### 1. Validação de qualidade entre as camadas Bronze e Silver do Lakehouse

In [0]:
%python
tabelas = ["orders", "customers", "products", "sellers", "order_items", "order_payments"]

resumo = []

for t in tabelas:
    bronze_count = spark.table(f"olist_project.bronze.{t}").count()
    silver_count = spark.table(f"olist_project.silver.{t}").count()
    descartadas = bronze_count - silver_count
    pct_descartado = round((descartadas / bronze_count) * 100, 2) if bronze_count > 0 else 0

    resumo.append({
        "tabela": t,
        "bronze_linhas": bronze_count,
        "silver_linhas": silver_count,
        "linhas_descartadas": descartadas,
        "pct_descartado": pct_descartado,
    })

resumo_df = spark.createDataFrame(resumo)
resumo_df.createOrReplaceTempView("resumo_qualidade")

display(resumo_df)


**O que o código faz**

Ele percorre estas 6 tabelas:

`orders`
`customers`
`products`
`sellers`
`order_items`
`order_payments`

Para cada uma, compara a** quantidade de registros na Bronze com a quantidade na Silver**.

**Por que fizemos isso?**

Para ter uma visão rápida do impacto das transformações da Silver.

| Cálculo          | O que representa                           |
| ---------------- | ------------------------------------------ |
| `bronze_count`   | Total de linhas que chegaram na Bronze     |
| `silver_count`   | Total de linhas que permaneceram na Silver |
| `descartadas`    | Linhas que não chegaram à Silver           |
| `pct_descartado` | Percentual de linhas descartadas           |


orders: bronze_linhas=99441, silver_linhas=99446, descartadas=-5

Repare: descartadas deu negativo (-5), o que significa o oposto de "descartado". A tabela Silver tem 5 linhas a mais que a Bronze, não a menos.

Por que isso é esperado (e não um erro)

Lembra da Etapa 4(`02_silver_transform`)? Nós simulamos um batch incremental e usamos MERGE INTO pra inserir 5 pedidos fictícios novos (sim_001 a sim_005) direto na tabela Silver.

Esses 5 registros nunca existiram na Bronze, eles foram criados artificialmente, direto na Silver, como parte da simulação. Por isso a Silver ficou com 99.446 linhas (99.441 originais + 5 simulados), enquanto a Bronze continua com 99.441 (porque a Bronze nunca foi tocada pelo MERGE INTO).

Este comportamento é comportamento correto e esperado, dado o que fizemos de propósito. Mas é um ótimo exemplo de por que esse notebook de qualidade é valioso: se você não soubesse a história por trás (a simulação da Etapa 4), esse número negativo pareceria estranho ou até um bug. Documentar essa explicação no próprio notebook evita confusão futura.

Nota:
> **Nota sobre `orders`:** o valor negativo em `linhas_descartadas` (-5) é esperado — não indica erro.
> Reflete os 5 pedidos simulados (`sim_001` a `sim_005`) inseridos via `MERGE INTO` na Etapa 4,
> que existem na Silver mas nunca passaram pela Bronze.

In [0]:
SELECT * FROM resumo_qualidade ORDER BY pct_descartado DESC

### 2. Validação de unicidade

Checagem consolidada de duplicatas por chave em todas as tabelas Silver
Verificando se as chaves que definimos como identificadoras de cada tabela estão realmente únicas.

In [0]:
-- Checagem consolidada de duplicatas por chave em todas as tabelas Silver
SELECT 'orders' as tabela, COUNT(*) as duplicatas
FROM (SELECT order_id FROM olist_project.silver.orders GROUP BY order_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'customers', COUNT(*)
FROM (SELECT customer_id FROM olist_project.silver.customers GROUP BY customer_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'products', COUNT(*)
FROM (SELECT product_id FROM olist_project.silver.products GROUP BY product_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'sellers', COUNT(*)
FROM (SELECT seller_id FROM olist_project.silver.sellers GROUP BY seller_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'order_items', COUNT(*)
FROM (SELECT order_id, order_item_id FROM olist_project.silver.order_items GROUP BY order_id, order_item_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'order_payments', COUNT(*)
FROM (SELECT order_id, payment_sequential FROM olist_project.silver.order_payments GROUP BY order_id, payment_sequential HAVING COUNT(*) > 1)

### 3. Completude das chaves

Checagem de nulos nas colunas-chave, camada Silver

validação de completude das colunas-chave da camada Silver

Existem registros na Silver que estão sem os identificadores necessários para relacionar as tabelas?

In [0]:
-- Checagem de nulos nas colunas-chave, camada Silver
SELECT 'orders' as tabela, COUNT(*) as nulos_chave
FROM olist_project.silver.orders WHERE order_id IS NULL OR customer_id IS NULL
UNION ALL
SELECT 'customers', COUNT(*)
FROM olist_project.silver.customers WHERE customer_id IS NULL
UNION ALL
SELECT 'products', COUNT(*)
FROM olist_project.silver.products WHERE product_id IS NULL
UNION ALL
SELECT 'sellers', COUNT(*)
FROM olist_project.silver.sellers WHERE seller_id IS NULL
UNION ALL
SELECT 'order_items', COUNT(*)
FROM olist_project.silver.order_items WHERE order_id IS NULL
UNION ALL
SELECT 'order_payments', COUNT(*)
FROM olist_project.silver.order_payments WHERE order_id IS NULL

Como interpretar o resultado esperado

1. A primeira query (resumo Bronze vs Silver) deve mostrar linhas_descartadas baixo ou zero pra maioria - grandes descartes indicariam algo errado na limpeza
2. A segunda query (duplicatas) deve vir toda zerada - já validamos isso individualmente, aqui é só a confirmação consolidada
3. A terceira query (nulos) também deve vir toda zerada - mesma lógica